## Video 2: read a file, check its probe

__Plan__:
1. Read a raw data file
2. Investigate its probe
3. Attach a probe to a recording in many different ways

__Resources__:

Recording fomats supported in spikeinterface: https://spikeinterface.readthedocs.io/en/latest/modules/extractors.html#raw-data-formats

Install plotting widget backends: https://spikeinterface.readthedocs.io/en/latest/modules/widgets.html

DANDI datasets: https://dandiarchive.org/dandiset 

ProbeInterface: https://github.com/SpikeInterface/probeinterface \
ProbeInterface library: https://github.com/SpikeInterface/probeinterface_library

Advanced probe tutorial: https://github.com/SpikeInterface/SpikeInterface-Training-Edinburgh-May24/tree/main/hands_on/probe_handling \
Video of tutorial: https://youtu.be/pHze_8s4Qak?si=eqb4QpwYbo_YRbr3&t=1062

## Reading a recording

During all of these tutorials, I highly recommend you try and use one of your own recordings. If you don't have one: you can get some from DANDI. Here's some tetrode data: https://dandiarchive.org/dandiset/000943 Or find something you're interested in by searching DANDI.

The recording I'll be using is an `OpenEphys` recording, so I open it using `si.read_openephys`. You can check out the formats that SpikeInterface (through Python-Neo https://github.com/NeuralEnsemble/python-neo) can read here: https://spikeinterface.readthedocs.io/en/latest/modules/extractors.html#raw-data-formats

In [ ]:
import spikeinterface.full as si

In [ ]:
path_to_np2_recording = "path/to/recording/"

In [ ]:
recording = si.read_openephys(path_to_np2_recording)

If you're lucky, your recording format will automatically contain the probe data. You can check by running the following code. If the recording does have a probe attached, we can plot it using a widget (more about installing widgets https://spikeinterface.readthedocs.io/en/latest/modules/widgets.html), as follows:

In [ ]:
recording.has_probe()

In [ ]:
%matplotlib widget
si.plot_probe_map(recording)

If this looks ok for your recording, feel free to move on to the next tutorial. If it doesn't, or you want to learn more about probes, keep going!

## A recording without a probe, but with contact positions

Sometimes a recording doesn't have a probe attached, but **does** have the locations of the probes. This is the case for the recordings contained in https://dandiarchive.org/dandiset/000943. Like all DANDI datasets, these recording will be in the NeuroDataWithoutBorder format. We can read these files using `si.read_nwb`.

In [ ]:
recording_nwb = si.read_nwb("/path/to/sub-M11_ses-M11-D1-2021-05-10-10-34-08_ecephys.nwb")
recording_nwb.has_probe()

This outputs `False`, but if we plot the probe we get a sensible result because the probe's contact locations are stored correctly.

In [ ]:
si.plot_probe_map(recording_nwb)

Now suppose the recording does not have the probe info. What to do depends on your situation. We'll no go through a few possible scenarios.

## A probe from a metadata file

Some recording formats spit out a metadata file. For example, SpikeGLX gives `.meta` files. If you have these, you can load the probe using `ProbeInterface` and then attach it to your recording like so:

In [ ]:
import probeinterface as pi
probe = pi.read_spikeglx('/path/to/joystick_g0_t0.imec0.ap.meta')
recording_with_probe = recording_without_probe.set_probe(probe)

## A probe from a manufacturer

Some probes are "stored" in the ProbeInterface Library (https://github.com/SpikeInterface/probeinterface_library). If you're using one of these, you can load, attach and visualise it like so:

In [ ]:
recording_64, _ = si.generate_ground_truth_recording(num_channels=4)

import probeinterface as pi

manufacturer = 'cambridgeneurotech'
probe_name = 'ASSY-77-E-1'

probe = pi.get_probe(manufacturer, probe_name)
probe.wiring_to_device(pathway="ASSY-77>Adpt.A64-Om32_2x-sm-NN>RHD2164")

probe

In [ ]:
recording_with_probe = recording_64.set_probe(probe)
si.plot_probe_map(recording_with_probe)

## Do It Yourself

If all else fails, you can do it yourself. ProbeInterface has great documentation on this (https://probeinterface.readthedocs.io/en/main/examples/ex_01_generate_probe_from_sratch.html)

Here's an example of making your own tetrode:

In [ ]:
recording_tetrode, _ = si.generate_ground_truth_recording(num_channels=4)

import numpy as np
n = 4
positions = np.array( [ [0,0], [0,20], [20,0], [20,20] ] )

probe = pi.Probe()
probe.set_contacts(positions=positions, shapes='circle', shape_params={'radius': 7.5})
probe.set_device_channel_indices(channel_indices=[0,1,2,3])
probe.create_auto_shape()

recording_tetrode = recording_tetrode.set_probe(probe)

In [ ]:
si.plot_probe_map(recording_tetrode)